# ROSA+ Extended with CRF and spaCy Syntactic Features
This notebook demonstrates extending ROSA+ with:
- Conditional Random Fields for sequence labeling
- spaCy dependency parsing as hidden meta-state
- Deep syntactic understanding for text relations

In [1]:
# Install dependencies
%pip install spacy sklearn-crfsuite torch numpy
!python -m spacy download en_core_web_sm


[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 24.4 MB/s eta 0:00:00 0:00:01

[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [2]:
from rosaplus import ROSAPlus
import spacy
import sklearn_crfsuite
from sklearn_crfsuite import metrics
import torch
import torch.nn as nn
import numpy as np
from typing import List, Dict, Tuple, Optional
from collections import defaultdict
import json

## 1. Syntactic Feature Extractor using spaCy

In [3]:
class SpacySyntacticFeatureExtractor:
    """Extract syntactic features from text using spaCy dependency parsing."""
    
    def __init__(self, model_name='en_core_web_sm'):
        self.nlp = spacy.load(model_name)
        
    def extract_token_features(self, text: str) -> List[Dict[str, any]]:
        """Extract per-token syntactic features including dependency tree structure."""
        doc = self.nlp(text)
        features = []
        
        for token in doc:
            feature_dict = {
                'text': token.text,
                'lemma': token.lemma_,
                'pos': token.pos_,
                'tag': token.tag_,
                'dep': token.dep_,
                'head_pos': token.head.pos_,
                'head_dep': token.head.dep_,
                'is_root': token.dep_ == 'ROOT',
                'n_lefts': len(list(token.lefts)),
                'n_rights': len(list(token.rights)),
                'is_alpha': token.is_alpha,
                'is_stop': token.is_stop,
                'shape': token.shape_,
                # Subtree features for context
                'subtree_span': len(list(token.subtree)),
                'ancestors': [anc.dep_ for anc in token.ancestors][:3],  # Max 3 ancestors
                'children_deps': [child.dep_ for child in token.children],
            }
            features.append(feature_dict)
        
        return features
    
    def extract_sentence_syntax_tree(self, text: str) -> List[Dict]:
        """Extract full dependency tree structure for each sentence."""
        doc = self.nlp(text)
        trees = []
        
        for sent in doc.sents:
            tree = {
                'text': sent.text,
                'root': sent.root.text,
                'root_pos': sent.root.pos_,
                'tokens': len(sent),
                'edges': [(token.text, token.dep_, token.head.text) 
                         for token in sent if token.head != token],
                'noun_chunks': [chunk.text for chunk in sent.noun_chunks],
            }
            trees.append(tree)
        
        return trees

## 2. CRF-Enhanced ROSA+ Model

In [4]:
class ROSAPlusCRF:
    """ROSA+ extended with CRF for sequence labeling with syntactic features."""
    
    def __init__(self, max_order: int = 1048576, use_eot: bool = False):
        self.rosa = ROSAPlus(max_order=max_order, use_eot=use_eot, seed=42)
        self.syntax_extractor = SpacySyntacticFeatureExtractor()
        self.crf = sklearn_crfsuite.CRF(
            algorithm='lbfgs',
            c1=0.1,
            c2=0.1,
            max_iterations=100,
            all_possible_transitions=True
        )
        self.char_to_syntax_map = {}  # Maps character positions to syntactic features
        
    def _token_to_crf_features(self, tokens: List[Dict], idx: int) -> Dict[str, any]:
        """Convert token features to CRF feature format with context window."""
        token = tokens[idx]
        features = {
            'bias': 1.0,
            'token.lower': token['text'].lower(),
            'token.pos': token['pos'],
            'token.tag': token['tag'],
            'token.dep': token['dep'],
            'token.head_pos': token['head_pos'],
            'token.is_root': token['is_root'],
            'token.shape': token['shape'],
            'token.is_alpha': token['is_alpha'],
        }
        
        # Add context from previous token
        if idx > 0:
            prev = tokens[idx - 1]
            features.update({
                '-1:token.lower': prev['text'].lower(),
                '-1:token.pos': prev['pos'],
                '-1:token.dep': prev['dep'],
            })
        else:
            features['BOS'] = True
        
        # Add context from next token
        if idx < len(tokens) - 1:
            next_tok = tokens[idx + 1]
            features.update({
                '+1:token.lower': next_tok['text'].lower(),
                '+1:token.pos': next_tok['pos'],
                '+1:token.dep': next_tok['dep'],
            })
        else:
            features['EOS'] = True
        
        # Add dependency tree features
        if token['ancestors']:
            features['ancestor_path'] = '_'.join(token['ancestors'])
        if token['children_deps']:
            features['children'] = '_'.join(token['children_deps'])
        
        return features
    
    def train(self, texts: List[str], labels: Optional[List[List[str]]] = None):
        """Train ROSA+ base model and CRF with syntactic features."""
        print("Training ROSA+ base model...")
        for text in texts:
            self.rosa.train_example(text)
        self.rosa.build_lm()
        
        print("\nExtracting syntactic features for CRF...")
        X_train = []
        y_train = []
        
        for i, text in enumerate(texts):
            token_features = self.syntax_extractor.extract_token_features(text)
            sent_features = [self._token_to_crf_features(token_features, j) 
                           for j in range(len(token_features))]
            X_train.append(sent_features)
            
            # If labels provided, use them; otherwise create dummy labels
            if labels and i < len(labels):
                y_train.append(labels[i])
            else:
                # Create auto-labels based on POS tags
                auto_labels = [feat['pos'] for feat in token_features]
                y_train.append(auto_labels)
        
        print("\nTraining CRF model...")
        self.crf.fit(X_train, y_train)
        print("Training complete!")
    
    def predict_with_syntax(self, text: str) -> Dict:
        """Generate predictions using both ROSA+ and CRF with syntactic features."""
        # Get ROSA+ base distribution
        rosa_dist = self.rosa.get_dist(text[:min(len(text), 100)])  # Use last 100 chars as context
        
        # Get syntactic features and CRF predictions
        token_features = self.syntax_extractor.extract_token_features(text)
        if token_features:
            crf_features = [self._token_to_crf_features(token_features, i) 
                          for i in range(len(token_features))]
            crf_labels = self.crf.predict([crf_features])[0]
            crf_marginals = self.crf.predict_marginals([crf_features])[0]
        else:
            crf_labels = []
            crf_marginals = []
        
        # Get dependency tree structure
        syntax_trees = self.syntax_extractor.extract_sentence_syntax_tree(text)
        
        return {
            'rosa_distribution': rosa_dist,
            'crf_labels': crf_labels,
            'crf_marginals': crf_marginals,
            'syntax_trees': syntax_trees,
            'token_features': token_features,
        }
    
    def generate_syntax_aware(self, prompt: str, max_tokens: int = 100, 
                             temperature: float = 0.7) -> str:
        """Generate text using ROSA+ refined by syntactic CRF constraints."""
        generated = []
        current_text = prompt
        
        for _ in range(max_tokens):
            # Get predictions with syntactic features
            predictions = self.predict_with_syntax(current_text)
            rosa_dist = predictions['rosa_distribution']
            
            # Sample from ROSA+ distribution
            if rosa_dist:
                chars = list(rosa_dist.keys())
                probs = np.array(list(rosa_dist.values()))
                # Apply temperature
                probs = np.power(probs, 1.0 / temperature)
                probs = probs / probs.sum()
                next_char = np.random.choice(chars, p=probs)
            else:
                break
            
            generated.append(next_char)
            current_text += next_char
            
            # Check if we completed a sentence based on syntax
            if next_char in '.!?\n':
                syntax_trees = predictions['syntax_trees']
                if syntax_trees and len(syntax_trees) > 0:
                    # We have a complete syntactic structure
                    pass
        
        return ''.join(generated)
    
    def save(self, base_path: str):
        """Save the hybrid model."""
        self.rosa.save(f"{base_path}_rosa.json")
        # Save CRF
        import pickle
        with open(f"{base_path}_crf.pkl", 'wb') as f:
            pickle.dump(self.crf, f)
    
    def load(self, base_path: str):
        """Load the hybrid model."""
        self.rosa = ROSAPlus.load(f"{base_path}_rosa.json")
        import pickle
        with open(f"{base_path}_crf.pkl", 'rb') as f:
            self.crf = pickle.load(f)

## 3. Neural CRF with GRU for Deep Syntactic Understanding

In [5]:
class BiGRUCRF(nn.Module):
    """Neural CRF with Bidirectional GRU for deeper syntactic modeling."""
    
    def __init__(self, vocab_size: int, tag_size: int, 
                 embedding_dim: int = 100, hidden_dim: int = 256):
        super(BiGRUCRF, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.gru = nn.GRU(embedding_dim, hidden_dim // 2, 
                         num_layers=2, bidirectional=True, batch_first=True)
        self.hidden2tag = nn.Linear(hidden_dim, tag_size)
        
        # CRF transition parameters
        self.transitions = nn.Parameter(torch.randn(tag_size, tag_size))
        self.start_transitions = nn.Parameter(torch.randn(tag_size))
        self.end_transitions = nn.Parameter(torch.randn(tag_size))
        
    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        """Forward pass returning emission scores."""
        embeds = self.embedding(x)
        gru_out, _ = self.gru(embeds)
        emissions = self.hidden2tag(gru_out)
        return emissions
    
    def viterbi_decode(self, emissions: torch.Tensor) -> List[int]:
        """Viterbi decoding for best path."""
        seq_len, tag_size = emissions.shape
        scores = torch.zeros(seq_len, tag_size)
        backpointers = torch.zeros(seq_len, tag_size, dtype=torch.long)
        
        # Initialize with start transitions
        scores[0] = emissions[0] + self.start_transitions
        
        # Forward pass
        for t in range(1, seq_len):
            broadcast_scores = scores[t-1].unsqueeze(1)
            broadcast_transitions = self.transitions.unsqueeze(0)
            next_scores = broadcast_scores + broadcast_transitions + emissions[t].unsqueeze(0)
            scores[t], backpointers[t] = torch.max(next_scores, dim=0)
        
        # Add end transitions
        scores[-1] += self.end_transitions
        
        # Backtrack
        best_path = [torch.argmax(scores[-1]).item()]
        for t in range(seq_len - 1, 0, -1):
            best_path.insert(0, backpointers[t, best_path[0]].item())
        
        return best_path

## 4. Complete Hybrid System: ROSA+ + CRF + spaCy

In [6]:
class ROSAPlusSyntaxCRFHybrid:
    """Complete system combining ROSA+, neural CRF, and spaCy syntax trees."""
    
    def __init__(self, max_order: int = 1048576):
        self.rosa_crf = ROSAPlusCRF(max_order=max_order)
        self.syntax_state_memory = {}  # Cache syntax states
        self.transition_graph = defaultdict(list)  # Track syntax-aware transitions
        
    def train(self, texts: List[str]):
        """Train the complete hybrid system."""
        print("=" * 60)
        print("Training ROSA+ Syntax-CRF Hybrid System")
        print("=" * 60)
        
        # Train base ROSA+ and CRF
        self.rosa_crf.train(texts)
        
        # Build syntax transition graph
        print("\nBuilding syntax-aware transition graph...")
        for text in texts:
            trees = self.rosa_crf.syntax_extractor.extract_sentence_syntax_tree(text)
            for tree in trees:
                # Store syntax patterns
                root_pos = tree['root_pos']
                for edge in tree['edges']:
                    child, dep, parent = edge
                    self.transition_graph[(parent, dep)].append((child, root_pos))
        
        print(f"Built transition graph with {len(self.transition_graph)} unique patterns")
    
    def generate_with_syntax_constraints(self, prompt: str, 
                                        max_tokens: int = 200,
                                        enforce_syntax: bool = True) -> str:
        """Generate text with syntactic coherence constraints."""
        current_text = prompt
        generated = []
        
        for i in range(max_tokens):
            # Get comprehensive predictions
            predictions = self.rosa_crf.predict_with_syntax(current_text)
            
            # Extract current syntactic state
            if predictions['syntax_trees']:
                current_tree = predictions['syntax_trees'][-1]
                current_root = current_tree['root_pos']
            else:
                current_root = None
            
            # Get ROSA+ distribution
            rosa_dist = predictions['rosa_distribution']
            
            if not rosa_dist:
                break
            
            # Apply syntactic constraints if enabled
            if enforce_syntax and current_root:
                # Boost probabilities of syntactically valid continuations
                valid_patterns = self.transition_graph.get((current_root, 'any'), [])
                # This is a simplified constraint - in practice, you'd implement
                # more sophisticated syntax-aware reweighting
            
            # Sample next character
            chars = list(rosa_dist.keys())
            probs = np.array(list(rosa_dist.values()))
            probs = probs / probs.sum()
            next_char = np.random.choice(chars, p=probs)
            
            generated.append(next_char)
            current_text += next_char
        
        return ''.join(generated)
    
    def analyze_syntax_aware_prediction(self, text: str) -> Dict:
        """Deep analysis of text with all syntactic and statistical features."""
        predictions = self.rosa_crf.predict_with_syntax(text)
        
        analysis = {
            'text': text,
            'rosa_next_char_probs': predictions['rosa_distribution'],
            'crf_sequence_labels': predictions['crf_labels'],
            'dependency_trees': predictions['syntax_trees'],
            'token_syntactic_features': predictions['token_features'],
        }
        
        # Add interpretation
        if predictions['syntax_trees']:
            tree = predictions['syntax_trees'][-1]
            analysis['syntactic_summary'] = {
                'sentence_root': tree['root'],
                'sentence_structure': tree['root_pos'],
                'complexity': tree['tokens'],
                'noun_phrases': tree['noun_chunks'],
            }
        
        return analysis

## 5. Practical Example: Training and Using the System

In [11]:
# Sample training data
import requests

# Train on Shakespare
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

# Download the text
response = requests.get(url)
training_texts = response.text.split("\n")


# Initialize and train the hybrid system
print("Initializing ROSA+ Syntax-CRF Hybrid...\n")
hybrid_model = ROSAPlusSyntaxCRFHybrid(max_order=1024)
hybrid_model.train(training_texts)

Initializing ROSA+ Syntax-CRF Hybrid...

Training ROSA+ Syntax-CRF Hybrid System
Training ROSA+ base model...


100%|██████████| 45/45 [00:00<00:00, 714938.18it/s]



Extracting syntactic features for CRF...

Training CRF model...
Training complete!

Building syntax-aware transition graph...
Built transition graph with 51073 unique patterns


In [15]:
# Test: Analyze a new text
test_text = "The machine learning model processes natural language"
analysis = hybrid_model.analyze_syntax_aware_prediction(test_text)

print("\n" + "="*60)
print("SYNTACTIC ANALYSIS RESULTS")
print("="*60)
print(f"\nInput Text: {test_text}")
print(f"\nNext Character Predictions (ROSA+):")
for char, prob in sorted(analysis['rosa_next_char_probs'].items(), 
                        key=lambda x: x[1], reverse=True)[:5]:
    print(f"  '{char}': {prob:.4f}")

if 'syntactic_summary' in analysis:
    print(f"\nSyntactic Structure:")
    print(f"  Root: {analysis['syntactic_summary']['sentence_root']}")
    print(f"  Type: {analysis['syntactic_summary']['sentence_structure']}")
    print(f"  Noun Phrases: {analysis['syntactic_summary']['noun_phrases']}")

print(f"\nCRF Sequence Labels: {analysis['crf_sequence_labels'][:10]}...")


SYNTACTIC ANALYSIS RESULTS

Input Text: The machine learning model processes natural language

Next Character Predictions (ROSA+):
  ' ': 0.3433
  '.': 0.2324
  '!': 0.1531
  ';': 0.1524
  's': 0.0807

Syntactic Structure:
  Root: processes
  Type: VERB
  Noun Phrases: ['The machine learning model', 'natural language']

CRF Sequence Labels: ['DET' 'NOUN' 'VERB' 'NOUN' 'VERB' 'ADJ' 'NOUN']...


In [17]:
# Test: Generate syntax-aware text
prompt = "The neural network "
generated = hybrid_model.generate_with_syntax_constraints(
    prompt, 
    max_tokens=100, 
    enforce_syntax=True
)

print("\n" + "="*60)
print("SYNTAX-AWARE TEXT GENERATION")
print("="*60)
print(f"\nPrompt: {prompt}")
print(f"Generated: {generated}")

# Analyze the generated text
full_text = prompt + generated
gen_analysis = hybrid_model.analyze_syntax_aware_prediction(full_text)
print(f"\nGenerated Text Syntax Trees:")
for i, tree in enumerate(gen_analysis['dependency_trees'][:3]):
    print(f"  Sentence {i+1}: Root='{tree['root']}' ({tree['root_pos']}), Tokens={tree['tokens']}")


SYNTAX-AWARE TEXT GENERATION

Prompt: The neural network 
Generated: the king, what with lamented. First, forsooth, but by mighty King of Naples, his appppmaapppdppppppp

Generated Text Syntax Trees:
  Sentence 1: Root='network' (NOUN), Tokens=10
  Sentence 2: Root='appppmaapppdppppppp' (NOUN), Tokens=13


## 6. Advanced: Visualizing Syntax-Aware State Transitions

In [18]:
def visualize_syntax_state(text: str, model: ROSAPlusSyntaxCRFHybrid):
    """Visualize how syntax features influence ROSA+ states."""
    analysis = model.analyze_syntax_aware_prediction(text)
    
    print("\nDETAILED SYNTAX-STATE MAPPING:")
    print("-" * 60)
    
    for i, token_feat in enumerate(analysis['token_syntactic_features'][:10]):
        print(f"\nToken {i}: '{token_feat['text']}'")
        print(f"  POS: {token_feat['pos']:<10} DEP: {token_feat['dep']:<10}")
        print(f"  Head: {token_feat['head_pos']:<10} Root: {token_feat['is_root']}")
        print(f"  Children: {token_feat['children_deps']}")
        if i < len(analysis['crf_sequence_labels']):
            print(f"  CRF Label: {analysis['crf_sequence_labels'][i]}")

# Visualize example
visualize_syntax_state(
    "The sophisticated algorithm efficiently processes complex linguistic structures.",
    hybrid_model
)


DETAILED SYNTAX-STATE MAPPING:
------------------------------------------------------------

Token 0: 'The'
  POS: DET        DEP: det       
  Head: PROPN      Root: False
  Children: []
  CRF Label: DET

Token 1: 'sophisticated'
  POS: ADJ        DEP: amod      
  Head: PROPN      Root: False
  Children: []
  CRF Label: ADJ

Token 2: 'algorithm'
  POS: PROPN      DEP: nsubj     
  Head: VERB       Root: False
  Children: ['det', 'amod']
  CRF Label: PROPN

Token 3: 'efficiently'
  POS: ADV        DEP: advmod    
  Head: VERB       Root: False
  Children: []
  CRF Label: ADV

Token 4: 'processes'
  POS: VERB       DEP: ROOT      
  Head: VERB       Root: True
  Children: ['nsubj', 'advmod', 'dobj', 'punct']
  CRF Label: VERB

Token 5: 'complex'
  POS: ADJ        DEP: amod      
  Head: NOUN       Root: False
  Children: []
  CRF Label: ADJ

Token 6: 'linguistic'
  POS: ADJ        DEP: amod      
  Head: NOUN       Root: False
  Children: []
  CRF Label: ADJ

Token 7: 'structures'
  P

## 7. Save and Load the Model

In [19]:
# Save the trained model
model_path = "rosa_syntax_crf_hybrid"
hybrid_model.rosa_crf.save(model_path)
print(f"Model saved to {model_path}_rosa.json and {model_path}_crf.pkl")

# Load the model
loaded_model = ROSAPlusSyntaxCRFHybrid()
loaded_model.rosa_crf.load(model_path)
print("Model loaded successfully!")

Model saved to rosa_syntax_crf_hybrid_rosa.json and rosa_syntax_crf_hybrid_crf.pkl


Model loaded successfully!
